# Week 2 environment and data check

Run this notebook before Week 2 to confirm that the Python environment can create, audit, calculate, and plot the artificial data used for return practice. It does not download market data and does not introduce features, signals, models, or backtests.

**Completion conditions before Week 2**

- Run all cells from top to bottom without an error.
- Import the required packages and display their versions.
- Create and audit a small artificial OHLCV table.
- Calculate simple returns, log returns, and a wealth index.
- Display a price-and-wealth figure and preserve visible evidence.

The artificial values are for calculation practice only. They do not describe a security and cannot support an investment claim. In Colab, select `Runtime > Run all`. If execution fails, preserve the complete error message and the number of the first failing cell.

## 1. Check Python and the required packages

Week 2 requires `numpy` and `pandas`; this environment check also uses `matplotlib` for one figure. The common market, assets, data provider, and redistribution conditions are still pending, so an online-data package is not required here.

In [ ]:
import sys, platform, importlib.util

print('Python:', sys.version)
print('Platform:', platform.platform())

required = ['numpy', 'pandas', 'matplotlib']

missing = []
for pkg in required:
    status = 'OK' if importlib.util.find_spec(pkg) else 'MISSING'
    print(f'{pkg:12s} required {status}')
    if status == 'MISSING':
        missing.append(pkg)

if missing:
    raise ModuleNotFoundError(f'Install the missing packages before Week 2: {missing}')

## 2. Import the packages

Import the required packages only after the previous cell confirms that they are available. Record the displayed versions with your environment evidence.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

print('numpy:', np.__version__)
print('pandas:', pd.__version__)
print('matplotlib:', matplotlib.__version__)

## 3. Create the artificial price table

This table is fixed inside the notebook so that every student receives the same result. `Adj_Close` is an artificial analysis field; no corporate action occurs in this sample.

In [ ]:
plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['axes.grid'] = True

records = [
    ['2026-01-05', 100.0, 101.5, 99.5, 100.0, 100.0, 1_000_000],
    ['2026-01-06', 101.0, 103.0, 100.5, 102.0, 102.0, 1_100_000],
    ['2026-01-07', 102.0, 102.5, 100.0, 101.0, 101.0, 950_000],
    ['2026-01-08', 101.0, 104.5, 100.8, 104.0, 104.0, 1_250_000],
    ['2026-01-09', 104.0, 104.2, 102.0, 103.0, 103.0, 980_000],
    ['2026-01-12', 104.0, 107.0, 103.5, 106.0, 106.0, 1_400_000],
]
columns = ['Date', 'Open', 'High', 'Low', 'Close', 'Adj_Close', 'Volume']
prices = pd.DataFrame(records, columns=columns)
prices['Date'] = pd.to_datetime(prices['Date'], format='%Y-%m-%d', errors='raise')
prices = prices.set_index('Date').sort_index()

high_violation = prices['High'] < prices[['Open', 'Close']].max(axis=1)
low_violation = prices['Low'] > prices[['Open', 'Close']].min(axis=1)
range_violation = prices['Low'] > prices['High']
audit = pd.Series({
    'rows': len(prices),
    'date_is_unique': prices.index.is_unique,
    'date_is_sorted': prices.index.is_monotonic_increasing,
    'missing_values': int(prices.isna().sum().sum()),
    'nonpositive_prices': int((prices[['Open', 'High', 'Low', 'Close', 'Adj_Close']] <= 0).sum().sum()),
    'negative_volume': int((prices['Volume'] < 0).sum()),
    'weekend_rows': int((prices.index.dayofweek >= 5).sum()),
    'ohlc_violations': int((high_violation | low_violation | range_violation).sum()),
})

assert prices.shape == (6, 6)
assert prices.index.is_unique and prices.index.is_monotonic_increasing
assert (prices[['Open', 'High', 'Low', 'Close', 'Adj_Close']] > 0).all().all()
assert (prices['Volume'] >= 0).all()
assert (prices.index.dayofweek < 5).all()
assert audit['ohlc_violations'] == 0
print(prices)
print('\nAudit results')
print(audit)

## 4. Calculate and check returns

Use the artificial analysis price to calculate simple returns, log returns, and the value of one initial unit. The first return remains missing because there is no earlier price.

In [ ]:
result = pd.DataFrame(index=prices.index)
result['simple_return'] = prices['Adj_Close'].pct_change(fill_method=None)
result['log_return'] = np.log(prices['Adj_Close'] / prices['Adj_Close'].shift(1))
result['wealth_index'] = (1 + result['simple_return'].fillna(0)).cumprod()

assert pd.isna(result['simple_return'].iloc[0])
assert np.allclose(
    result['simple_return'].dropna(),
    np.expm1(result['log_return'].dropna()),
)
assert np.isclose(
    result['wealth_index'].iloc[-1],
    prices['Adj_Close'].iloc[-1] / prices['Adj_Close'].iloc[0],
)
print(result.round(6))

## 5. Check figure output

The upper panel shows the artificial price level, and the lower panel shows the corresponding wealth index. Their different vertical scales must remain labeled.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
prices['Adj_Close'].plot(ax=axes[0], title='Artificial analysis price')
result['wealth_index'].plot(ax=axes[1], title='Wealth index from simple returns')
axes[0].set_ylabel('price units')
axes[1].set_ylabel('initial value = 1')
plt.tight_layout()
plt.show()

print('Price rows:', len(prices))
print('Non-missing returns:', int(result['simple_return'].notna().sum()))
print('Final wealth index:', round(result['wealth_index'].iloc[-1], 6))

## 6. Optional CSV export

CSV export is not required for the environment check. Leave `SAVE_OUTPUTS=False` unless you want to verify that the current notebook session can write a file.

In [ ]:
from pathlib import Path

SAVE_OUTPUTS = False
if SAVE_OUTPUTS:
    out_dir = Path('week2_environment_output')
    out_dir.mkdir(exist_ok=True)
    output_file = out_dir / 'week2_artificial_returns.csv'
    prices.join(result).to_csv(output_file)
    print('Saved:', output_file.resolve())
else:
    print('No file written. Set SAVE_OUTPUTS=True only for the optional export check.')

## 7. Completion record

Before Week 2, confirm each statement with visible output from this notebook:

- I can open the notebook and run every cell in order.
- I recorded the Python, NumPy, pandas, and matplotlib versions.
- I obtained six price rows, zero audit violations, five non-missing returns, and a final wealth index of 1.06.
- I can see both labeled panels of the figure.
- I understand that the artificial output is calculation evidence, not evidence about a market or future performance.
- If execution failed, I preserved the first complete error message and cell number.